In [67]:
instructions

"\nYou are a clinical data assistant. Your task is to:\n    1. Summarize the patient's historical medical records concisely.\n    2. Analyze 'Vitals' data (BP, Heart Rate, SpO2).\n    3. ALERT the user if you see dangerous trends:\n       - Increasing Blood Pressure (Hypertension progression).\n       - Tachycardia (Rising Heart Rate at rest).\n       - Dropping oxygen levels.\n    Provide a 'Clinical Summary' section and a 'Vitals Alert' section.\n\n"

In [66]:
from prompts import instructions

In [48]:
import asyncio
from agents import Agent, Runner, OpenAIChatCompletionsModel
from typing import List
from openai import AsyncOpenAI 
from dotenv import load_dotenv
import os

In [49]:
from openai import AsyncOpenAI 

load_dotenv(override=True)

api_key = os.getenv('API_KEY1')
base_url = os.getenv('BASE_URL')

print(f"API_KEY: {api_key}")
print(f"BASE_URL: {base_url}")

client = AsyncOpenAI(base_url=base_url, api_key=api_key)

model= OpenAIChatCompletionsModel(
                    model="openai.o4-mini",
                    openai_client=client
                )  

API_KEY: UcK175nqL61u4e7MyqJEw1cicuhBYYhW5z6jFTDb
BASE_URL: https://openai.generative.engine.capgemini.com/v1


In [50]:
instructions="""
You are a clinical data assistant. Your task is to:
    1. Summarize the patient's historical medical records concisely.
    2. Analyze 'Vitals' data (BP, Heart Rate, SpO2).
    3. ALERT the user if you see dangerous trends:
       - Increasing Blood Pressure (Hypertension progression).
       - Tachycardia (Rising Heart Rate at rest).
       - Dropping oxygen levels.
    Provide a 'Clinical Summary' section and a 'Vitals Alert' section.

"""

In [51]:
instructions

"\nYou are a clinical data assistant. Your task is to:\n    1. Summarize the patient's historical medical records concisely.\n    2. Analyze 'Vitals' data (BP, Heart Rate, SpO2).\n    3. ALERT the user if you see dangerous trends:\n       - Increasing Blood Pressure (Hypertension progression).\n       - Tachycardia (Rising Heart Rate at rest).\n       - Dropping oxygen levels.\n    Provide a 'Clinical Summary' section and a 'Vitals Alert' section.\n\n"

In [52]:
summarizer_agent = Agent(
        name="summarizer_agent",
        instructions=instructions,
        model=model
)

In [53]:
patient_medical_record ="""
Patient: John Doe, 58yo Male.
Records:
- 2026-01-10: Routine checkup. BP 120/80, HR 72. Healthy.
- 2026-02-15: Follow up. BP 135/85, HR 78. Patient reports stress.
- 2026-04-01: Urgent care visit. BP 155/95, HR 92. Complaining of headaches.
- 2026-04-20: Today. BP 162/100, HR 98.
"""

In [ ]:
patient_medical_record2 ="""
    { "patient_id": "p-100001",
      "demographics": { "age": 68, "gender": "male" },
      "encounters": [ 
        { "encounter_id": "enc-9001", "start": "2024-09-15T10:15:00Z", "end": "2024-09-15T10:50:00Z", "vitals": [ { "type": "heart_rate", "value": 78, "unit": "beats/min", "timestamp": "2024-09-15T10:20:00Z" }, { "type": "blood_pressure_systolic", "value": 120, "unit": "mmHg", "timestamp": "2024-09-15T10:22:00Z" }, { "type": "blood_pressure_diastolic", "value": 78, "unit": "mmHg", "timestamp": "2024-09-15T10:22:00Z" }, { "type": "temperature", "value": 37.0, "unit": "C", "timestamp": "2024-09-15T10:25:00Z" }, { "type": "oxygen_saturation", "value": 98, "unit": "%", "timestamp": "2024-09-15T10:28:00Z" } ], "notes": "Bedside monitor connected; data quality: good." },
        { "encounter_id": "enc-9002", "start": "2024-09-22T14:00:00Z", "end": "2024-09-22T15:30:00Z", "vitals": [ { "type": "heart_rate", "value": 92, "unit": "beats/min", "timestamp": "2024-09-22T14:15:00Z" }, { "type": "blood_pressure_systolic", "value": 118, "unit": "mmHg", "timestamp": "2024-09-22T14:25:00Z" }, { "type": "blood_pressure_diastolic", "value": 76, "unit": "mmHg", "timestamp": "2024-09-22T14:25:00Z" }, { "type": "temperature", "value": 38.3, "unit": "C", "timestamp": "2024-09-22T14:35:00Z" }, { "type": "oxygen_saturation", "value": 94, "unit": "%", "timestamp": "2024-09-22T14:40:00Z" } ], "notes": "New antibiotic started; vitals show rising HR and fever." } ] }
"""

In [55]:
result =await Runner.run(summarizer_agent, patient_medical_record)

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [56]:
print(type(result))

<class 'agents.result.RunResult'>


In [57]:
result2 =await Runner.run(summarizer_agent, patient_medical_record2)

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [58]:
print(type(result2))

<class 'agents.result.RunResult'>


In [64]:

print(result.final_output)
print("-------------------------------")
print(result2.final_output)

Clinical Summary  
- 58-year-old male with no significant issues at January visit (BP 120/80, HR 72)  
- February follow-up shows rising BP (135/85) and mild HR increase (78) alongside reported stress  
- Early April urgent care visit for headaches; BP 155/95, HR 92  
- Today’s reading continues upward trend: BP 162/100, HR 98  

Vitals Alert  
- Blood Pressure: Progressive increase from 120/80 → 135/85 → 155/95 → 162/100.  
  • Alert: Now in Stage 2 hypertension range; elevated risk for cardiovascular events.  
- Heart Rate: Steady rise from 72 → 78 → 92 → 98 bpm.  
  • Warning: Approaching tachycardia threshold; monitor for persistent resting tachycardia.  
- Oxygen Saturation (SpO2): Not recorded in available data.  
  • Recommendation: Obtain SpO2 measurements to rule out hypoxemia.
-------------------------------
Clinical Summary  
Patient p-100001 is a 68-year-old male with two recent inpatient encounters one week apart.  
- 2024-09-15: Routine monitoring showed normal vital sign

In [79]:

import nest_asyncio
nest_asyncio.apply()

from summarizer import summarize_medical_record


summary = summarize_medical_record(patient_medical_record)


OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [80]:
print(summary)

Clinical Summary  
• Patient: John Doe, 58-year-old male  
• Medical history: Progressive elevation in blood pressure over 3 months  
  – 2026-01-10: BP 120/80 mm Hg (normal), HR 72 bpm  
  – 2026-02-15: BP 135/85 mm Hg (prehypertension), HR 78 bpm; reports increased stress  
  – 2026-04-01: BP 155/95 mm Hg (stage 1 hypertension), HR 92 bpm; complaints of headaches  
  – 2026-04-20: BP 162/100 mm Hg (stage 2 hypertension), HR 98 bpm  
• No SpO₂ readings documented  

Vitals Alert  
• Hypertension progression:  
  – Systolic BP rose from 120 to 162 mm Hg, diastolic from 80 to 100 mm Hg in 3 months  
  – Current BP in stage 2 hypertension range – requires prompt management  
• Rising heart rate:  
  – HR increased from 72 bpm to 98 bpm at rest – approaching tachycardia  
• SpO₂ not recorded – recommend measuring to rule out hypoxemia  

Recommendation: Consider urgent cardiovascular evaluation, initiate or adjust antihypertensive therapy, address stress management, and obtain SpO₂.
